# 02 · EHR Features — PLAN_end_to_end A3–A8 + E1

Consumes `cohort_v2/cohort_base.parquet` from `01_cohort_selection.ipynb` and emits
`cohort_study_level.parquet` (§A8). **This is where the old cohort died**, so every
failure mode gets an explicit guard:

| Old-cohort failure | Root cause | Guard here |
|---|---|---|
| `frac_studies_with_future_organism = 0.448` | join on `subject_id`, no time anchor | §E1 asserts == 0 (C9) |
| mean 94.85 micro rows/study (max 1427) | Cartesian product | dedupe → specimen → study (C4); assert median 0–2 |
| `CANCELLED` = 30.9% of orgs | inner join dropped NULL, inflating the rest | LEFT joins only; C3 asserts NULL ≈ 0.59 |
| 59% NO_GROWTH silently gone | `org_name` NULL + `dropna` defaults | status derived at **specimen level** from NULL |

**Two clocks, and they mean different things (§A3):**
- `charttime` = specimen **collected** → clinical *suspicion*
- `storetime` = result **available** → the *result*

That gap is not noise — it is why `frac_specimen_sent_before = 78.8%` while
`frac_organism_known_before = 43.2%`. The 35-point difference is **legitimate
signal** (Tier 2), not leakage.

**Deviation from §A3 (deliberate, see C2):** the plan's uniform "date fallback at
end-of-day + `time_imprecise=1`" is not applied to `storetime`. Inventing a result
time creates a traceless artifact; a null `storetime` (~0.8%) means we can assert
neither "resulted" nor "pending", so those specimens are **excluded outright**.
`charttime` nulls (~7.6%) gate only Tier 2, where "a specimen was sent" is far less
hour-sensitive — there the `chartdate` fallback + `t2_time_imprecise` flag stays.


## C0 · Config

In [ ]:
import os, re, numpy as np, pandas as pd

PROJECT = 'mimic-ehr-v310'
HOSP    = 'physionet-data.mimiciv_3_1_hosp'
TMP_DS  = 'ehr_fusion_tmp'

try:
    from google.colab import auth, drive
    auth.authenticate_user(); drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT, location='US')

def run_q(sql, label=''):
    """Dry-run, then execute.

    A dry-run makes BigQuery fully parse + name-resolve the query for **0 bytes
    billed**. It catches `Unrecognized name: x` / typos before a single byte is
    scanned, and prints what the real run will cost. Every query in this notebook
    goes through here -- these CTEs are long enough that a silent typo is likelier
    than not.
    """
    dry = client.query(sql, job_config=bigquery.QueryJobConfig(
        dry_run=True, use_query_cache=False))
    print(f'  [{label}] dry-run OK · will scan {dry.total_bytes_processed/1e9:.2f} GB')
    return client.query(sql).to_dataframe()

def need(*frag_tokens):
    """Guard against stale SQL fragments left in the kernel.

    MICRO_CTE / SX are plain strings built in C2 and interpolated by later cells.
    Re-running a *consumer* cell after an edit silently reuses whatever string is
    still in memory, and BigQuery then reports `Unrecognized name: x` pointing at
    a line the file on disk does not contain. This turns that into a clear
    'go re-run C2'.

    Usage:  need(('SX', 'ct_clamped'), ('MICRO_CTE', 'AS ct_clamped'))
    """
    for name, token in frag_tokens:
        frag = globals().get(name)
        if frag is None:
            raise RuntimeError(f'{name} is not defined -> run cell C2 first.')
        if token not in frag:
            raise RuntimeError(
                f'{name} in memory is STALE (missing {token!r}).\n'
                f'    -> re-run cell C2 (SQL fragments), then this cell.\n'
                f'    Runtime > Restart and run all is the safe option.')

OUT_DIR = os.path.join(DATA_DIR, 'cohort_v2')
assert os.path.exists(os.path.join(OUT_DIR,'cohort_base.parquet')), \
    'run 01_cohort_selection.ipynb first'

# --- §A3 windows ---
LAB_WINDOW_H   = 24      # labs: [index_time - 24h, index_time]; coverage saturates after 24h
T1_WINDOW_DAYS = 7       # fresh result. CLINICAL choice (~one antibiotic course), NOT a
                         # data optimum: marginal coverage decays smoothly with no elbow
                         # (2.00 -> 1.06 -> 0.69 -> 0.53 pp/day over 3->5->7->10->14).
T3_WINDOW_DAYS = 90      # recent history; >90d goes to Tier 4 (aggregate only)

# --- §A6 reserved ids (fixed) ---
PAD, UNK, MISSING, NO_GROWTH, CANCELLED, NEGATIVE = 0, 1, 2, 3, 4, 5
N_RESERVED = 6           # preliminary tokens start here, species after them

# §A6 tier-2 tokens: real signal at coarser resolution -> own ids, NOT dropped
PRELIM_RE = re.compile(r'GRAM\s+(POSITIVE|NEGATIVE)|MIXED\s+BACTERIAL|^POSITIVE|AFB\s+GROWN'
                       r'|PRESUMPTIVE|^MOLD$|^YEAST$|NON[\s-]?FERMENTER|ANAEROBIC', re.I)
ADMIN_RE  = re.compile(r'^CANCELLED$|^NEGATIVE$|ISOLATE', re.I)

# §A7 names six; lab_rbc is added (Tuan) -- it was in the old dataset and dropping it on
# an ASSERTED collinearity with hemoglobin would be a guess. C7 measures corr(rbc,hgb) and
# prints it, so the redundancy is a number, not an opinion.
LABS = ['lab_bun','lab_creatinine','lab_glucose','lab_hemoglobin','lab_platelet','lab_wbc',
        'lab_rbc']
WINSOR_PCT = (0.5, 99.5)
# rbc is a red-cell count (~4.5 M/uL, roughly symmetric) like hemoglobin -> winsorize,
# do NOT log. The log set is the right-skewed ones (wbc >100 in leukaemia is real signal).
LOG_LABS   = {'lab_wbc','lab_platelet','lab_glucose','lab_creatinine','lab_bun'}

# recency features -- same value+mask contract as the labs (§A7): NEVER impute and
# discard the mask. 58.8% of T1 studies have no positive culture, so a positive-only
# recency feature is NULL for the majority; the any-specimen one covers them.
RECENCY = ['t1_days_since_last_specimen', 't1_days_since_last_positive',
           't3_days_since_last_positive']
print('OUT_DIR', OUT_DIR)

## C1 · Load cohort from 01 and push it to BigQuery

We upload the **exact table 01 emitted** rather than re-deriving the cohort in SQL.
A re-derivation would silently drift the moment 01's inclusion rules change, and
the regenerated split only exists in that file. This is our own 227k×7 table, not
the old mystery CSV.

In [ ]:
import json as _json
cohort = pd.read_parquet(os.path.join(OUT_DIR,'cohort_base.parquet'))

# Read the label list 01 DECLARED; never re-derive it by exclusion. Guessing
# "everything that isn't in this hardcoded list is a label" silently promotes any
# new column 01 adds into a disease label -- which is exactly what happened when
# n_frontal/n_lateral/n_view_ok were added: 02 built a 16-class class_freq.pkl,
# 3 of whose entries were image counts, and nothing complained.
META_PATH = os.path.join(OUT_DIR, 'cohort_meta.json')
assert os.path.exists(META_PATH), f'{META_PATH} missing -> re-run 01'
_meta = _json.load(open(META_PATH))
LABEL_COLS = _meta['labels']
assert len(LABEL_COLS) == _meta['n_labels'] == 13, \
    f'expected 13 labels, meta says {_meta["n_labels"]}: {LABEL_COLS}'
missing = [c for c in LABEL_COLS if c not in cohort.columns]
assert not missing, f'cohort_base.parquet is missing declared labels {missing} -> re-run 01'
assert _meta['n_studies'] == len(cohort), \
    f'cohort_meta.json says {_meta["n_studies"]:,} studies but the parquet has ' \
    f'{len(cohort):,} -> 01 emitted them at different times, re-run 01'

print(f'cohort: {len(cohort):,} studies | {len(LABEL_COLS)} labels (declared by 01) | '
      f'splits {cohort.split.value_counts().to_dict()}')
print(f'  labels: {LABEL_COLS}')
print(f'  non-label cols carried through: '
      f'{[c for c in cohort.columns if c not in LABEL_COLS]}')

keys = cohort[['study_id','subject_id','index_time','cur_hadm','cur_edstay','setting','split']].copy()
keys['index_time'] = pd.to_datetime(keys['index_time'])
for c in ['study_id','subject_id','cur_hadm','cur_edstay']:
    keys[c] = keys[c].astype('Int64')

ds = f'{PROJECT}.{TMP_DS}'
client.create_dataset(bigquery.Dataset(ds), exists_ok=True)
client.load_table_from_dataframe(keys, f'{ds}.coh',
    job_config=bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE')).result()
COH = f'`{ds}.coh`'
print('uploaded ->', f'{ds}.coh')

## C2 · §A3 — the two clocks, handled by their **sensitivity**, not one rule

The plan's blanket "fallback to date at end-of-day + `time_imprecise=1`" is
replaced. **Deviation from §A3, deliberate** — end-of-day assignment *invents a
timestamp*: it manufactures pseudo-times piled at day boundaries which then
interact with the `index_time` distribution, producing an artifact nobody can
trace later. The two clocks differ in how much a missing time costs:

| Clock | Null | What it gates | Decision |
|---|---|---|---|
| `storetime` (result available) | **~0.8%** | T1/T3/T4 (`result known`) **and** T2 (pending needs `index_time < storetime`) | **Drop the specimen.** Without it we can assert neither "resulted" nor "pending" — it is unusable in every tier. Costs <1%. |
| `charttime` (specimen collected) | **~7.6%** | T2 only (`sent`) | **`chartdate` fallback + flag.** "A specimen was sent" is far less hour-sensitive than "the result was back". |

Where the fallback is used, `chartdate` resolves to **end-of-day**, which is the
*conservative* direction for T2: we only claim suspicion existed if the entire day
preceded the film. `t2_time_imprecise` is emitted so this subset is ablatable.

Dropping is only safe if it isn't **selective** — if `storetime` were
disproportionately null for no-growth specimens, we'd be re-deleting the 59% by a
new route. C2-bis checks that before the drop is applied.

In [ ]:
# ---- audit the drop BEFORE applying it (is it selective?) ----
audit = client.query(f"""
SELECT
  CASE WHEN org_itemid IS NULL THEN 'NO_GROWTH(null org)'
       WHEN UPPER(org_name)='CANCELLED' THEN 'CANCELLED'
       WHEN UPPER(org_name)='NEGATIVE'  THEN 'NEGATIVE'
       ELSE 'POSITIVE' END                          AS status,
  COUNT(*)                                          AS n_rows,
  ROUND(COUNTIF(storetime IS NULL)/COUNT(*),4)      AS frac_storetime_null,
  ROUND(COUNTIF(charttime IS NULL)/COUNT(*),4)      AS frac_charttime_null,
  ROUND(COUNTIF(chartdate IS NULL)/COUNT(*),4)      AS frac_chartdate_null
FROM `{HOSP}.microbiologyevents` GROUP BY status ORDER BY n_rows DESC
""").to_dataframe()
print('null rates by status — is the storetime drop selective?')
print(audit.to_string(index=False))
spread = audit.frac_storetime_null.max() - audit.frac_storetime_null.min()
print(f'\nspread of storetime-null across statuses: {spread:.4f}')
if spread > 0.05:
    print('!! WARNING: storetime nulls concentrate in one status -> dropping is SELECTIVE.')

In [ ]:
MICRO_CTE = f"""
mres AS (
  SELECT
    m.subject_id, m.hadm_id, m.micro_specimen_id,
    m.org_itemid, m.org_name, m.spec_itemid, m.spec_type_desc,
    CAST(m.storetime AS TIMESTAMP)                                   AS st,   -- exact, never faked
    -- charttime, or chartdate@end-of-day when null. CLAMPED to st: a specimen
    -- cannot be collected after its own result is available. Without the clamp the
    -- end-of-day fallback invents ct > st, which then trips the T2 leak assert on
    -- tiers that never even read ct.
    LEAST(
      COALESCE(CAST(m.charttime AS TIMESTAMP),
               TIMESTAMP_ADD(CAST(CAST(m.chartdate AS DATE) AS TIMESTAMP),
                             INTERVAL 86399 SECOND)),
      CAST(m.storetime AS TIMESTAMP)
    )                                                                AS ct,
    CAST(m.charttime IS NULL AS INT64)                               AS ct_imprecise,
    -- did the clamp actually bite on THIS ROW? (must be measured here, at row
    -- level: after sstat's MIN() a single clamped row is invisible)
    CAST(COALESCE(CAST(m.charttime AS TIMESTAMP),
                  TIMESTAMP_ADD(CAST(CAST(m.chartdate AS DATE) AS TIMESTAMP),
                                INTERVAL 86399 SECOND))
         > CAST(m.storetime AS TIMESTAMP) AS INT64)                  AS ct_clamped
  FROM `{HOSP}.microbiologyevents` m
  WHERE m.storetime IS NOT NULL          -- EXCLUDE (~0.8%): cannot assert resulted OR pending
),
sstat AS (                               -- §A5 level 2: status lives at the SPECIMEN
  SELECT subject_id, hadm_id, micro_specimen_id,
    ANY_VALUE(spec_itemid)                        AS spec_itemid,
    MIN(ct) AS ct, MIN(st) AS st,
    MAX(ct_imprecise)                             AS ct_imprecise,
    MAX(ct_clamped)                               AS ct_clamped,
    ARRAY_AGG(DISTINCT org_itemid IGNORE NULLS)   AS org_itemids,
    CASE
      WHEN COUNTIF(org_itemid IS NOT NULL
                   AND UPPER(org_name) NOT IN ('CANCELLED','NEGATIVE')) > 0 THEN 'POSITIVE'
      WHEN COUNTIF(UPPER(org_name) = 'CANCELLED') > 0                       THEN 'CANCELLED'
      WHEN COUNTIF(UPPER(org_name) = 'NEGATIVE')  > 0                       THEN 'NEGATIVE'
      ELSE 'NO_GROWTH'                    -- all org_itemid NULL  <- 59% of rows
    END AS status
  FROM mres
  WHERE ct IS NOT NULL                    -- no charttime AND no chartdate -> unusable for T2
  GROUP BY subject_id, hadm_id, micro_specimen_id
)"""

# ---- sx: cohort x specimen, with the causal predicates. Defined HERE, next to
# MICRO_CTE, so that re-running this one cell refreshes every SQL fragment. Keeping
# them in separate cells is how a stale string survives an edit and the same query
# fails at the same character twice.
SX = f"""
sx AS (
  SELECT c.study_id, c.index_time, s.micro_specimen_id, s.spec_itemid,
         s.status, s.org_itemids, s.hadm_id, s.ct_imprecise, s.ct_clamped,
         s.st, s.ct,                     -- keep the raw clocks: C5 audits them directly
         -- storetime is exact for every surviving specimen -> a plain comparison.
         (s.st <= c.index_time)                                  AS result_known,
         -- ct may come from chartdate@end-of-day; that is conservative for "sent".
         (s.ct <= c.index_time)                                  AS specimen_sent,
         TIMESTAMP_DIFF(c.index_time, s.st, HOUR)/24.0           AS days_before,
         (c.cur_hadm IS NOT NULL AND s.hadm_id = c.cur_hadm)     AS same_enc
  FROM {COH} c JOIN sstat s ON s.subject_id = c.subject_id
)"""

# fail loudly if this cell ever runs against a stale edit
for _n, _t in [('MICRO_CTE','AS ct_clamped'), ('SX','s.ct_clamped'), ('SX','s.st, s.ct')]:
    assert _t in (MICRO_CTE if _n=='MICRO_CTE' else SX), f'{_n} built without {_t!r}'
print('SQL fragments (re)defined: MICRO_CTE, SX  -- consumers guarded by need()')

t = client.query(f"""
WITH {MICRO_CTE}
SELECT COUNT(*) AS n_specimens_kept,
  ROUND(AVG(ct_imprecise),4)                   AS frac_ct_from_chartdate,
  COUNTIF(status='POSITIVE')                   AS n_positive,
  COUNTIF(status='NO_GROWTH')                  AS n_no_growth,
  COUNTIF(status='CANCELLED')                  AS n_cancelled,
  COUNTIF(status='NEGATIVE')                   AS n_negative
FROM sstat
""").to_dataframe().iloc[0]
print('per-specimen status after the storetime exclusion (§A6):')
print(t.to_string())

## C3 · §E1 gate — NO_GROWTH must still be there (raw row level)

> §A6: *"**Do not search for the string.** MIMIC never writes `"NO GROWTH"` into
> `org_name`. Counting text finds nothing and leads to 'MIMIC has no negative
> cultures' — which is exactly how 59% of the data disappears unnoticed."*

`org_is_NULL`, `org_itemid_NULL`, `isolate_num_NULL` should be **the same number**
(2,352,859 ≈ 59%) — that identity is the documented "no growth ⇒ remaining columns
NULL" behaviour. If this assert fails, stop: something upstream is dropping NULLs.

In [ ]:
raw = client.query(f"""
SELECT COUNT(*) AS total_rows,
  COUNT(DISTINCT micro_specimen_id)            AS n_specimens,
  COUNTIF(org_itemid IS NULL)                  AS org_itemid_null,
  COUNTIF(org_name  IS NULL)                   AS org_name_null,
  COUNTIF(isolate_num IS NULL)                 AS isolate_num_null,
  COUNTIF(UPPER(org_name)='CANCELLED')         AS n_cancelled,
  COUNTIF(UPPER(org_name)='NEGATIVE')          AS n_negative,
  COUNT(DISTINCT org_itemid)                   AS distinct_org_itemid
FROM `{HOSP}.microbiologyevents`
""").to_dataframe().iloc[0]
print(raw.to_string())

frac_null = raw.org_itemid_null / raw.total_rows
print(f'\norg_itemid NULL fraction = {frac_null:.4f}  (§E1 expects ~0.59)')
print(f'rows/specimen = {raw.total_rows/raw.n_specimens:.2f}  (§A5 expects ~2.07)')
print(f'CANCELLED = {raw.n_cancelled/raw.total_rows:.4%}  (§A6: 1.04% raw, NOT the old 30.9%)')

assert abs(frac_null - 0.59) < 0.02, f'NO_GROWTH not preserved: {frac_null:.4f}'
assert raw.org_itemid_null == raw.org_name_null == raw.isolate_num_null, \
    'the three NULL counts must be identical (no-growth => all NULL)'
print('\nOK  NO_GROWTH preserved at raw row level')

### C3-bis · Reconcile rows ↔ specimens — **59% and 84% are both right**

A row-level constant must never be asserted against a specimen-level quantity.
`org_name` NULL is 59% of **rows** (§A6), but that is *not* the expected NO_GROWTH
share of **specimens**, because the two populations fan out differently:

| specimen | fans out to | rows |
|---|---|---|
| **positive** | ~1 organism × ~12 antibiotics | many |
| **no-growth** | no organism, no antibiotics | ~1 |

Per row, positives are **over-represented**, so NO_GROWTH is *diluted* to 59%.
De-duplicating to `micro_specimen_id` (§A5) removes exactly that inflation, so the
NO_GROWTH share **rises** — ≈0.84 at specimen level is the *expected* result, and
**the lift is the evidence the dedupe ran**.

The failure signal is the opposite one: if specimen-level NO_GROWTH also came back
≈0.59, the row multiplication would still be in the data.

In [ ]:
n_spec   = int(t.n_specimens_kept); n_ng = int(t.n_no_growth)
n_res    = int(t.n_positive + t.n_cancelled + t.n_negative)
row_tot  = int(raw.total_rows);     row_null = int(raw.org_itemid_null)
row_null_frac, spec_ng_frac = row_null / row_tot, n_ng / n_spec
rows_per_ng  = row_null / max(n_ng, 1)                 # no antibiotic fan-out
rows_per_res = (row_tot - row_null) / max(n_res, 1)    # organism x antibiotic

print(f'rows      : {row_tot:,}   NULL {row_null:,} = {row_null_frac:.3f}')
print(f'specimens : {n_spec:,}   NO_GROWTH {n_ng:,} = {spec_ng_frac:.3f}')
print(f'dropped by storetime-null: {int(raw.n_specimens)-n_spec:,} '
      f'({1 - n_spec/int(raw.n_specimens):.2%})   <- §A3 exclusion, expect <1%')
print(f'\nrows per NO_GROWTH specimen : {rows_per_ng:.2f}   (expect ~1-2, no antibiotics)')
print(f'rows per resulted specimen  : {rows_per_res:.2f}   (expect ~5-7, antibiotic panel)')
print(f'rows per specimen overall   : {row_tot/n_spec:.2f}   (§A5 expects ~2.07)')

assert spec_ng_frac > row_null_frac, (
    f'specimen NO_GROWTH ({spec_ng_frac:.3f}) <= row NULL ({row_null_frac:.3f}) '
    '=> row multiplication NOT removed; the §A5 dedupe did not run')
assert rows_per_res > rows_per_ng, 'resulted specimens must fan out more than no-growth ones'
assert abs(row_tot/n_spec - 2.07) < 0.5, 'rows/specimen far from the §A5 reference 2.07'
print(f'\nOK  dedupe verified: NO_GROWTH {row_null_frac:.3f} (rows) -> '
      f'{spec_ng_frac:.3f} (specimens)')

if int(t.n_negative) < 1000:
    print(f'\n!! LOOK: org_name="NEGATIVE" is {int(raw.n_negative):,} rows but only '
          f'{int(t.n_negative):,} specimens.\n   Those rows sit on specimens that also grew a '
          'real organism, so POSITIVE wins the CASE.\n   Harmless for status, but it means '
          'NEGATIVE is a per-test result, not a per-specimen one.')

## C4 · §A4 — the four tiers, split by **recency** not encounter

**Deviation from §A3/§A4 (deliberate, measured).** The plan gates Tier 1 on *same
encounter* with no day bound, and Tier 3 on *prior encounter* ≤90d. Two problems:

1. **They sit on different axes and cannot be ordered.** A result 95 days old is
   excluded from T3 (prior encounter, out of window) but is T1 — "the strongest
   signal" — if the patient simply had not been discharged yet. Same result, same
   clinical value; the only difference is administrative.
2. **ED could never reach Tier 1.** `same_enc` compares against `cur_hadm`, which is
   NULL for ED-only and outpatient studies, so `has_t1` was **exactly 0.000** for
   both. Fixing that under the old scheme needs a `cur_edstay` join, which needs
   `microbiologyevents` to carry `stay_id` — unverified, and possibly false.

Re-splitting on recency **dissolves both**: no encounter join is needed at all, and
T1/T3/T4 become a genuine ordinal ladder.

| Tier | Condition (all `result_known`) | Emits |
|---|---|---|
| 1 | `days_before < 7` — any encounter | `t1_org_ids`, `t1_spec_ids`, `t1_status`, `t1_same_encounter` (flag), 2 recency features |
| 2 | sent, result **not** known | `t2_pending`, `t2_spec_ids`, `t2_n_pending` |
| 3 | `7 ≤ days_before < 90` — any encounter | `t3_org_ids`, `t3_days_since_last_positive` |
| 4 | `days_before ≥ 90` | aggregates **only** — no organism identity |

**Measured effect of the remap:** T1 `29.25% → 36.30%` (+7.05pp) — lost 1,493 studies
(2.44% of old T1, stale same-encounter) but gained 16,241 (fresh, non-same-encounter).
This is a **net expansion**, not a lossy cut. `ED T1: 0.0000 → 0.0695`,
`outpatient: 0.0000 → 0.0667`. ED's ~7% ceiling is physiology, not a residual bug —
ED cultures are sent during the visit and result 24–72h later, so at film time they
are *pending* (T2 = 31.2%). That is precisely why Tier 2 exists.

**`same_enc` survives as a flag, not a filter** — "same illness episode" is real
information. But it is a **strict subset of `setting=='inpatient'`** (that setting is
*defined* by `cur_hadm`), so it must stay independently ablatable and any gain from it
has to survive `setting` stratification (§C).

**Tier 2 never leaks the pending result** — only the fact a specimen was sent and what
type. **Tier 4 carries no `org_ids` by design** (§A4): >90d history routes to the label
through report-writing style rather than imaging, so it is stated *as utilization*.

In [ ]:
need(('MICRO_CTE','AS ct_clamped'), ('SX','s.ct_clamped'))

t1 = run_q(f"""
WITH {MICRO_CTE}, {SX}
SELECT study_id,
  ARRAY_AGG(DISTINCT o IGNORE NULLS)                       AS t1_org_itemids,
  ARRAY_AGG(DISTINCT spec_itemid IGNORE NULLS)             AS t1_spec_itemids,
  LOGICAL_OR(status='POSITIVE')                            AS t1_any_positive,
  LOGICAL_OR(status='NO_GROWTH')                           AS t1_any_no_growth,
  LOGICAL_OR(status='CANCELLED')                           AS t1_any_cancelled,
  COUNT(DISTINCT micro_specimen_id)                        AS t1_n_specimens,
  -- kept as a FLAG (was the filter): "same illness episode" is real information,
  -- but it is nested inside setting=='inpatient' -> must stay ablatable.
  LOGICAL_OR(same_enc)                                     AS t1_same_encounter,
  -- two recency features: 0.92d vs 6.9d inside the same window are not the same thing
  MIN(days_before)                                         AS t1_days_since_last_specimen,
  MIN(IF(status='POSITIVE', days_before, NULL))            AS t1_days_since_last_positive
FROM sx LEFT JOIN UNNEST(sx.org_itemids) o
WHERE result_known AND days_before >= 0 AND days_before < {T1_WINDOW_DAYS}
GROUP BY study_id
""", 'T1')
print('T1 studies:', len(t1))
print(f'  same-encounter share of T1 : {t1.t1_same_encounter.mean():.2%}')
print(f'  T1 with a POSITIVE culture : {t1.t1_days_since_last_positive.notna().mean():.2%}'
      f'   -> the rest are NO_GROWTH/CANCELLED only (hence the mask)')

t2 = run_q(f"""
WITH {MICRO_CTE}, {SX}
SELECT study_id,
  TRUE                                          AS t2_pending,
  ARRAY_AGG(DISTINCT spec_itemid IGNORE NULLS)  AS t2_spec_itemids,
  COUNT(DISTINCT micro_specimen_id)             AS t2_n_pending,
  -- share of this study's pending specimens whose send-time came from chartdate
  MAX(ct_imprecise)                             AS t2_time_imprecise
FROM sx
WHERE specimen_sent AND NOT result_known        -- sent, no result yet.  NO org identity.
GROUP BY study_id
""", 'T2')
print('T2 studies:', len(t2))
print('  of which send-time is date-only (ablatable):',
      f"{t2.t2_time_imprecise.mean():.2%}" if len(t2) else 'n/a')

t3 = run_q(f"""
WITH {MICRO_CTE}, {SX}
SELECT study_id,
  ARRAY_AGG(DISTINCT o IGNORE NULLS)            AS t3_org_itemids,
  MIN(IF(status='POSITIVE', days_before, NULL)) AS t3_days_since_last_positive,
  COUNT(DISTINCT micro_specimen_id)             AS t3_n_specimens
FROM sx LEFT JOIN UNNEST(sx.org_itemids) o
WHERE result_known
  AND days_before >= {T1_WINDOW_DAYS} AND days_before < {T3_WINDOW_DAYS}
GROUP BY study_id
""", 'T3')
print('T3 studies:', len(t3))

t4 = run_q(f"""
WITH {MICRO_CTE}, {SX}
SELECT study_id,
  COUNTIF(status='POSITIVE')                        AS t4_n_prior_positive_ever,
  MAX(days_before)/365.25                           AS t4_years_since_first_culture,
  COUNT(DISTINCT hadm_id)                           AS t4_n_prior_encounters
FROM sx
WHERE result_known AND days_before >= {T3_WINDOW_DAYS}
GROUP BY study_id
""", 'T4')
print('T4 studies:', len(t4))

## C5 · §E1 LEAKAGE GATE — fail fast, before anything else is built

§F step 3: *"Run E1 leakage asserts here (fail fast)."* Old-cohort reference values
that must all become **0**:

```
frac_collected_after_cxr        = 0.514
frac_resulted_after_cxr         = 0.672
frac_POSITIVE_resulted_after    = 0.689
frac_studies_with_future_organism = 0.448
```

In [ ]:
need(('MICRO_CTE','AS ct_clamped'), ('SX','s.ct_clamped'), ('SX','s.st, s.ct'))

leak = run_q(f"""
WITH {MICRO_CTE}, {SX},
org_used AS (  -- T1/T3: consume ORGANISM IDENTITY -> gated by st (result available).
               -- after the recency remap this is simply "resolved within 90 days".
  SELECT * FROM sx
  WHERE result_known AND days_before >= 0 AND days_before < {T3_WINDOW_DAYS}
),
t2_used AS (   -- T2: consumes only "a specimen was sent" -> gated by ct (collected)
  SELECT * FROM sx WHERE specimen_sent AND NOT result_known
),
t4_used AS (   -- T4: consumes counts only -> gated by st
  SELECT * FROM sx WHERE result_known AND days_before >= {T3_WINDOW_DAYS}
)
SELECT
  (SELECT COUNTIF(st > index_time) FROM org_used)                  AS n_org_resulted_after,
  (SELECT COUNTIF(ct > index_time) FROM t2_used)                   AS n_t2_collected_after,
  (SELECT COUNTIF(st > index_time) FROM t4_used)                   AS n_t4_resulted_after,
  (SELECT COUNT(DISTINCT IF(st > index_time, study_id, NULL))
     FROM org_used)                                                AS n_studies_future_org,
  (SELECT COUNT(DISTINCT study_id) FROM {COH})                     AS n_studies_total,
  -- diagnostic, NOT an assert: how often did the ct<=st clamp bite?
  (SELECT COUNTIF(ct_clamped = 1) FROM sx)                         AS n_ct_clamped_to_st,
  (SELECT COUNT(*) FROM sx)                                        AS n_sx_rows
""", 'leak-gate').iloc[0]

frac_future = leak.n_studies_future_org / leak.n_studies_total
print(f'T1/T3 organism identity resulted AFTER index_time : {leak.n_org_resulted_after}')
print(f'T2 specimen collected AFTER index_time            : {leak.n_t2_collected_after}')
print(f'T4 rows resulted AFTER index_time                 : {leak.n_t4_resulted_after}')
print(f'frac_studies_with_future_organism                 : {frac_future:.4f}   (old: 0.4480)')
print(f'\nct clamped to st (ct == st): {leak.n_ct_clamped_to_st:,} / {leak.n_sx_rows:,} '
      f'= {leak.n_ct_clamped_to_st/leak.n_sx_rows:.2%}   <- the end-of-day fallback biting')

# each tier is checked against the clock that ACTUALLY gates it -- checking ct on
# T1/T3/T4 would flag a fabricated timestamp those tiers never read.
assert leak.n_org_resulted_after == 0, 'LEAK: organism identity from the future'
assert leak.n_t2_collected_after  == 0, 'LEAK: T2 specimen collected after the film'
assert leak.n_t4_resulted_after   == 0, 'LEAK: T4 aggregate from the future'
assert frac_future == 0.0, f'LEAK: {frac_future:.4f} of studies see a future organism'
print('\nLEAKAGE GATE PASSED — all zero')

## C6 · §A6 vocab (train split only) + ragged ids

`org_itemid` / `spec_itemid` (integers) are the vocab keys, **not** strings — §A5:
avoids case/whitespace variants inflating `distinct_orgs = 718`.

Three tiers of token, not two — preliminary results are **real signal at coarser
resolution** (AFB ⇒ TB suspicion; MIXED ⇒ likely contaminated), so they get their
own ids instead of being lumped into `UNK`.

In [ ]:
names = run_q(f"""
SELECT org_itemid, ANY_VALUE(org_name) AS org_name, COUNT(*) AS n
FROM `{HOSP}.microbiologyevents` WHERE org_itemid IS NOT NULL
GROUP BY org_itemid
""", 'org-names')
id2name = dict(zip(names.org_itemid, names.org_name))

def classify(nm):
    if ADMIN_RE.search(nm or ''):  return 'admin'
    if PRELIM_RE.search(nm or ''): return 'prelim'
    return 'species'
names['kind'] = names.org_name.map(classify)
print(names.groupby('kind').agg(n_itemids=('org_itemid','size'), n_rows=('n','sum')).to_string())

train_ids = set(cohort.loc[cohort.split=='train','study_id'])
def train_tokens(df, col):
    from collections import Counter
    c = Counter()
    for lst in df.loc[df.study_id.isin(train_ids), col].dropna():
        c.update([int(x) for x in lst])
    return c

org_counts  = train_tokens(t1,'t1_org_itemids') + train_tokens(t3,'t3_org_itemids')
spec_counts = train_tokens(t1,'t1_spec_itemids') + train_tokens(t2,'t2_spec_itemids')

CANCELLED_IDS = {i for i,n in id2name.items() if (n or '').upper()=='CANCELLED'}
NEGATIVE_IDS  = {i for i,n in id2name.items() if (n or '').upper()=='NEGATIVE'}
prelim = [i for i in org_counts if classify(id2name.get(i,''))=='prelim']
specie = [i for i in org_counts if classify(id2name.get(i,''))=='species']
prelim.sort(key=lambda i:-org_counts[i]); specie.sort(key=lambda i:-org_counts[i])

org_vocab = {}
for i in CANCELLED_IDS: org_vocab[i] = CANCELLED
for i in NEGATIVE_IDS:  org_vocab[i] = NEGATIVE
nxt = N_RESERVED
for i in prelim: org_vocab[i] = nxt; nxt += 1
for i in specie: org_vocab[i] = nxt; nxt += 1
ORG_VOCAB_SIZE = nxt
spec_vocab = {i: k+N_RESERVED for k,(i,_) in
              enumerate(sorted(spec_counts.items(), key=lambda kv:-kv[1]))}
SPEC_VOCAB_SIZE = N_RESERVED + len(spec_vocab)

print(f'\norg vocab : {ORG_VOCAB_SIZE} ids = {N_RESERVED} reserved + '
      f'{len(prelim)} preliminary + {len(specie)} species   (train-only)')
print(f'spec vocab: {SPEC_VOCAB_SIZE} ids')
print('preliminary tokens kept as own ids:',
      [id2name.get(i) for i in prelim[:8]])

def to_ids(lst, vocab, empty):
    # a LEFT-merged study with no tier row arrives as NaN (a float), not []
    if lst is None or np.isscalar(lst) or not hasattr(lst, '__len__'): return [empty]
    if len(lst) == 0: return [empty]
    ids = sorted({vocab.get(int(x), UNK) for x in lst})
    return ids if ids else [empty]

## C7 · §A7 labs — 24h window, `storetime <= index_time`

§A7's six canonical labs **+ `lab_rbc`** (Tuan — it was in the old dataset). Most-recent
value in `[index_time − 24h, index_time]`. Coverage saturates after 24h (59.3% @6h →
**71.9% @24h** → 75.8% @7d), so a wider window buys little and stales the value.

**On `lab_rbc`:** the case against it is that it re-measures what hemoglobin already
measures. That is a *hypothesis*, not a fact about this cohort — so this cell prints
`corr(rbc, hgb)` and lets the number decide. They are not guaranteed redundant:
microcytic anaemia drops hemoglobin while the red-cell *count* holds up, so the pair
separates exactly where anaemia type matters. Keeping it costs 2 columns.

`fluid='Blood'` in the `lab_rbc` match is load-bearing — *"Red Blood Cells"* also exists
in **urine**, and counting urine RBCs as a blood count is the kind of error that produces
a plausible-looking column and no error message.

`lab_*_mask` is **never** dropped after imputation — whether a test was *ordered* is
itself a clinical variable (MNAR).

In [ ]:
labq = run_q(f"""
WITH lab_map AS (
  SELECT itemid, CASE
    WHEN LOWER(label) LIKE '%urea nitrogen%'    AND fluid='Blood' THEN 'lab_bun'
    WHEN LOWER(label) = 'creatinine'            AND fluid='Blood' THEN 'lab_creatinine'
    WHEN LOWER(label) = 'glucose'               AND fluid='Blood' THEN 'lab_glucose'
    WHEN LOWER(label) LIKE '%hemoglobin%'       AND fluid='Blood' THEN 'lab_hemoglobin'
    -- fluid='Blood' is load-bearing: 'Red Blood Cells' also exists in URINE (urinalysis),
    -- and counting urine RBCs as a blood count would be silent garbage. 'Nucleated Red
    -- Cells' does not match this pattern (no 'blood'), which is what we want.
    WHEN LOWER(label) LIKE '%red blood cell%'    AND fluid='Blood' THEN 'lab_rbc'
    WHEN LOWER(label) LIKE '%platelet%'                           THEN 'lab_platelet'
    WHEN LOWER(label) LIKE '%white blood cell%'                   THEN 'lab_wbc'
  END AS canon FROM `{HOSP}.d_labitems`
),
lr AS (
  SELECT le.subject_id, lm.canon, le.valuenum,
         CAST(le.charttime AS TIMESTAMP) AS ct,
         COALESCE(CAST(le.storetime AS TIMESTAMP),
                  CAST(le.charttime AS TIMESTAMP)) AS st
  FROM `{HOSP}.labevents` le JOIN lab_map lm USING (itemid)
  WHERE lm.canon IS NOT NULL AND le.valuenum IS NOT NULL AND le.charttime IS NOT NULL
),
j AS (
  SELECT c.study_id, lr.canon, lr.valuenum,
         TIMESTAMP_DIFF(c.index_time, lr.ct, MINUTE)/60.0 AS delta_h,
         ROW_NUMBER() OVER (PARTITION BY c.study_id, lr.canon
                            ORDER BY lr.ct DESC) AS rn
  FROM {COH} c JOIN lr ON lr.subject_id = c.subject_id
   AND lr.ct BETWEEN TIMESTAMP_SUB(c.index_time, INTERVAL {LAB_WINDOW_H} HOUR)
                 AND c.index_time
   AND lr.st <= c.index_time                    -- resulted, not merely drawn
)
SELECT study_id, canon, valuenum, delta_h FROM j WHERE rn = 1
""", 'labs')
print('lab rows:', len(labq))

val = labq.pivot(index='study_id', columns='canon', values='valuenum')
dlt = labq.pivot(index='study_id', columns='canon', values='delta_h')
have = val.reindex(cohort.study_id).notna()

# §A7's "71.9% @24h" is an ANY-lab figure. Per-lab coverage is a different
# quantity and is necessarily lower -- comparing one to the other is meaningless.
print('\nper-lab coverage (NOT comparable to the 71.9% reference):')
CBC = ['lab_hemoglobin','lab_platelet','lab_wbc','lab_rbc']   # ordered as one panel
BMP = ['lab_bun','lab_creatinine','lab_glucose']              # ordered as another
for grp, name in [(CBC,'CBC'), (BMP,'BMP/chem')]:
    for c in grp:
        if c in have:
            print(f'  {c:<18} {have[c].mean():7.2%}   [{name}]')
        else:
            print(f'  {c:<18} !! NOT MATCHED by the d_labitems CASE -> fix the label map')

any_lab = have.any(axis=1).mean()
n_lab   = have.sum(axis=1)
print(f'\nANY of the {len(LABS)} labs within {LAB_WINDOW_H}h : {any_lab:7.2%}'
      f'   <- THIS is what §A7 s 71.9% refers to')
print(f'labs per study: mean {n_lab.mean():.2f}/{len(LABS)} · '
      f'median {int(n_lab.median())} · zero labs {(n_lab==0).mean():.2%}')

# labs are ordered as panels, so the two groups should each be internally tight;
# a scattered spread inside a panel means the itemid map is wrong.
for grp, name in [(CBC,'CBC'), (BMP,'BMP/chem')]:
    g = [c for c in grp if c in have]
    if len(g) > 1:
        spread = have[g].mean().max() - have[g].mean().min()
        flag = 'OK' if spread < 0.05 else '!! check itemid map'
        print(f'  {name:<9} internal spread {spread:.2%}  {flag}')

# ---- does lab_rbc earn its place, or is it just hemoglobin again? MEASURE it. ----
# Both count red cells, so a high correlation is expected -- but "expected" is not
# "verified", and the old dataset carried rbc, so it gets tested rather than assumed.
if {'lab_rbc','lab_hemoglobin'} <= set(val.columns):
    pair = val[['lab_rbc','lab_hemoglobin']].dropna()
    r = pair['lab_rbc'].corr(pair['lab_hemoglobin'])
    print(f'\ncorr(lab_rbc, lab_hemoglobin) = {r:.3f}   on {len(pair):,} studies with both')
    if abs(r) > 0.95:
        print('   -> near-duplicate of hemoglobin; keep only if a downstream ablation shows a gain.')
    elif abs(r) > 0.80:
        print('   -> strongly collinear but not identical (anaemia type separates them:')
        print('      microcytic keeps rbc up while hgb falls). Cheap to keep, mask handles absence.')
    else:
        print('   -> LESS collinear than expected -> rbc carries its own signal, keep it.')
else:
    print('\n!! lab_rbc not in the pivot -> the d_labitems CASE matched nothing; fix the map.')

assert (labq['delta_h'] >= 0).all(), 'LEAK: lab measured after index_time'
print('\nOK  no lab from the future')

## C7-bis · VITALS — exact itemids, normalised to Celsius (§A7 extension)

Not in the plan. Added because `06`'s D3 put the EHR signal in **acute physiology**, and
fever/tachycardia is acute physiology more directly than creatinine is.

**The first version of this cell was wrong and the probe caught it.** Matching `d_items`
with `LOWER(label) LIKE '%heart rate%'` also selects **220046/220047 — the alarm
thresholds**, `'%o2 saturation%'` the SpO2 alarms, `'%respiratory rate%'` the ventilator
setting `Respiratory Rate (Set)`, and `'%systolic%'` the **arterial-line** BP as well as
the cuff.

**The damage is not visible in coverage** — which is what makes it dangerous. Measured,
the exact-itemid numbers land within 0.1pp of the LIKE ones for temp/HR/RR/SpO2: anyone
carrying an alarm threshold also carries a real reading, so the has/hasn't column is
unchanged. What the bug corrupts is the **value**. This query takes the *most recent* row
per vital, so an alarm limit of 120 charted after a heart rate of 88 lands in
`vital_heart_rate` as 120. Nothing asserts, nothing looks odd; the column is simply
wrong. (`sbp` is the one that moved — 53.3% → 47.7% — because arterial lines dropped out.)
Hence: **exact itemids, each pinned to its expected label**.

| vital | ICU itemid | ED column |
|---|---|---|
| temp (°C) | 223762 (C), 223761 (F→C) | `temperature` (**°F** → C) |
| heart_rate | 220045 | `heartrate` |
| resp_rate | 220210 | `resprate` |
| spo2 | 220277 | `o2sat` |
| sbp / dbp | 220179 / 220180 (non-invasive) | `sbp` / `dbp` |

**Units are measured, not assumed** — the probe read medians: `ed.vitalsign` 98.1 and
223761 98.4 (Fahrenheit); 223762 37.0 (Celsius). Everything is converted to **Celsius**.
Mixing them would produce a bimodal column centred on nothing, and no assert would fire.

**Clock:** labs need `storetime` (drawn ≠ resulted, §A3). A vital is known the moment it
is taken, and `ed.vitalsign` has no `storetime` at all — so ED gates on `charttime`, and
`chartevents` on `COALESCE(storetime, charttime)`, which can only be stricter.

**Implausible values are dropped, not winsorized.** A temperature of 0 °C is a typing
error, not a cold patient — unlike WBC > 100, which is real leukaemia and must survive
(that is why WBC is logged, not clipped). Bounds are physiologic; the count dropped is
printed.

**Three sources, not two.** `ed.vitalsign` alone reached only **58.9%** of ED studies —
not a broken join: it holds the *repeated* observations taken during a stay, and a short
visit may have none. **`ed.triage`** holds the arrival observations that *every* ED
patient gets. It carries no `charttime` because triage happens at `edstays.intime` by
definition, so that is its clock.

**Coverage cannot reach the labs' 67%**, and that is MIMIC, not a bug: `chartevents` is
the **ICU** system and MIMIC-IV has **no general-ward vitals**, while outpatients have
neither an ED nor an ICU stay. `mask=0` carries that honestly.

In [ ]:
# ---- resolve the datasets; do NOT guess the names (same discipline as 01 C1) ----
# Self-contained on purpose: this cell must run standalone after an edit, and a NameError
# from a fragment that lived in some earlier cell is exactly the stale-state trap that
# cost us four debugging rounds in this notebook already.
all_ds = [d.dataset_id for d in client.list_datasets('physionet-data')]
def _pick(pat, what):
    c = [d for d in all_ds if re.search(pat, d, re.I)]
    if not c:
        raise RuntimeError(f'no {what} dataset among {[d for d in all_ds if "mimic" in d.lower()]}')
    return f'physionet-data.{sorted(c)[-1]}'
ED  = _pick(r'mimic.*(_ed$|_ed_|iv_ed)', 'MIMIC-ED')
ICU = _pick(r'mimic.*iv.*_icu$',         'MIMIC-IV ICU')
print('ED  ->', ED, '| ICU ->', ICU)

vs_cols = [f.name for f in client.get_table(f'{ED}.vitalsign').schema]
assert 'storetime' not in vs_cols, 'ed.vitalsign gained a storetime -> revisit the clock choice'
assert 'temperature' in vs_cols and 'heartrate' in vs_cols, vs_cols

# ED has TWO vital sources and using only one loses ~40% of ED studies:
#   vitalsign -- repeated measurements DURING the stay (a short visit may have none)
#   triage    -- the arrival observations, which EVERY ED patient gets
# triage has no charttime; it happens at edstays.intime by definition, so that is its clock.
tr_cols = [f.name for f in client.get_table(f'{ED}.triage').schema]
print('triage cols:', tr_cols)
assert 'charttime' not in tr_cols, 'triage gained a charttime -> use it instead of edstays.intime'
for _c in ('temperature','heartrate','resprate','o2sat','sbp','dbp'):
    assert _c in tr_cols, f'triage missing {_c}'

# EXACT itemids -- never LIKE. See the header: LIKE also matches alarm limits and
# ventilator settings, which are charted constantly in the ICU.
IT_TEMP_C, IT_TEMP_F = 223762, 223761
IT_HR, IT_RR, IT_SPO2, IT_SBP, IT_DBP = 220045, 220210, 220277, 220179, 220180
VITALS = ['vital_temp_c','vital_heart_rate','vital_resp_rate','vital_spo2',
          'vital_sbp','vital_dbp']

# Pin the itemids to their labels. Hardcoded ids are only safe if they are checked: this
# is what stops '220046 Heart rate Alarm - High' from ever being read as a heart rate.
_EXPECT = {IT_TEMP_C:'temperature celsius', IT_TEMP_F:'temperature fahrenheit',
           IT_HR:'heart rate', IT_RR:'respiratory rate', IT_SPO2:'o2 saturation pulseoxymetry',
           IT_SBP:'non invasive blood pressure systolic',
           IT_DBP:'non invasive blood pressure diastolic'}
_chk = run_q(f"""SELECT itemid, label, unitname FROM `{ICU}.d_items`
                 WHERE itemid IN ({','.join(map(str,_EXPECT))}) ORDER BY itemid""", 'itemid-verify')
print(_chk.to_string(index=False))
assert len(_chk) == len(_EXPECT), f'{len(_chk)}/{len(_EXPECT)} itemids exist in d_items'
for _i, _l in zip(_chk.itemid, _chk.label):
    assert _EXPECT[int(_i)] == str(_l).strip().lower(), \
        f'itemid {_i} is "{_l}", expected "{_EXPECT[int(_i)]}" -> MIMIC renumbered, fix the map'
print('OK  every itemid resolves to its expected label (no alarm / ventilator-setting rows)')
# physiologic bounds: outside these a value is a documentation error, not a sick patient
VIT_RANGE = {'vital_temp_c':(25,45), 'vital_heart_rate':(10,300), 'vital_resp_rate':(2,80),
             'vital_spo2':(30,100), 'vital_sbp':(30,300), 'vital_dbp':(10,200)}

vitq = run_q(f"""
WITH ce AS (                                   -- ICU: exact itemids, F->C where needed
  SELECT c.subject_id,
         COALESCE(CAST(c.storetime AS TIMESTAMP), CAST(c.charttime AS TIMESTAMP)) AS ct,
         CASE c.itemid WHEN {IT_TEMP_C} THEN c.valuenum
                       WHEN {IT_TEMP_F} THEN (c.valuenum - 32) * 5/9 END AS vital_temp_c,
         IF(c.itemid = {IT_HR},   c.valuenum, NULL) AS vital_heart_rate,
         IF(c.itemid = {IT_RR},   c.valuenum, NULL) AS vital_resp_rate,
         IF(c.itemid = {IT_SPO2}, c.valuenum, NULL) AS vital_spo2,
         IF(c.itemid = {IT_SBP},  c.valuenum, NULL) AS vital_sbp,
         IF(c.itemid = {IT_DBP},  c.valuenum, NULL) AS vital_dbp
  FROM `{ICU}.chartevents` c
  WHERE c.itemid IN ({IT_TEMP_C},{IT_TEMP_F},{IT_HR},{IT_RR},{IT_SPO2},{IT_SBP},{IT_DBP})
    AND c.valuenum IS NOT NULL
),
ved AS (                                       -- ED: temperature is FAHRENHEIT (median 98.1)
  SELECT subject_id, CAST(charttime AS TIMESTAMP) AS ct,
         (temperature - 32) * 5/9 AS vital_temp_c,
         heartrate AS vital_heart_rate, resprate AS vital_resp_rate,
         o2sat     AS vital_spo2,       sbp      AS vital_sbp, dbp AS vital_dbp
  FROM `{ED}.vitalsign`
),
vtr AS (                                       -- ED triage: the arrival obs every patient gets.
  SELECT t.subject_id, CAST(e.intime AS TIMESTAMP) AS ct,   -- triage IS arrival; no charttime exists
         (t.temperature - 32) * 5/9 AS vital_temp_c,        -- Fahrenheit, like ed.vitalsign
         t.heartrate AS vital_heart_rate, t.resprate AS vital_resp_rate,
         t.o2sat     AS vital_spo2,       t.sbp      AS vital_sbp, t.dbp AS vital_dbp
  FROM `{ED}.triage` t JOIN `{ED}.edstays` e USING (subject_id, stay_id)
),
allv AS (SELECT * FROM ce UNION ALL SELECT * FROM ved UNION ALL SELECT * FROM vtr),
clean AS (                                     -- drop the impossible; keep the extreme
  SELECT subject_id, ct,
    {", ".join(f"IF(a.{v} BETWEEN {lo} AND {hi}, a.{v}, NULL) AS {v}"
               for v, (lo, hi) in VIT_RANGE.items())}
  FROM allv a
),
long AS (
  {" UNION ALL ".join(f"SELECT subject_id, ct, '{v}' AS vname, {v} AS val FROM clean WHERE {v} IS NOT NULL"
                      for v in VITALS)}
),
j AS (
  SELECT c.study_id, l.vname, l.val,
         TIMESTAMP_DIFF(c.index_time, l.ct, MINUTE)/60.0 AS delta_h,
         ROW_NUMBER() OVER (PARTITION BY c.study_id, l.vname ORDER BY l.ct DESC) AS rn
  FROM {COH} c JOIN long l
    ON l.subject_id = c.subject_id
   AND l.ct BETWEEN TIMESTAMP_SUB(c.index_time, INTERVAL {LAB_WINDOW_H} HOUR) AND c.index_time
)
SELECT study_id, vname, val, delta_h FROM j WHERE rn = 1
""", 'vitals')
print('vital rows:', len(vitq))

vval = vitq.pivot(index='study_id', columns='vname', values='val')
vdlt = vitq.pivot(index='study_id', columns='vname', values='delta_h')
vhave = vval.reindex(cohort.study_id).notna()

print(f'\nVITALS coverage in [index-{LAB_WINDOW_H}h, index]  (ICU + ED vitalsign + ED triage):')
_st = cohort.set_index('study_id').setting
for v in VITALS:
    if v not in vhave: print(f'  {v:18s} !! NOT MATCHED'); continue
    by = vhave[v].groupby(_st.reindex(vhave.index).values).mean()
    print(f'  {v:18s} ALL {vhave[v].mean():6.2%} | ' +
          ' | '.join(f'{k} {by.get(k,0):5.1%}' for k in ('inpatient','ed','outpatient')))

print(f'\nANY vital within {LAB_WINDOW_H}h : {vhave.any(axis=1).mean():7.2%}   '
      f'(labs any: 66.95%)')
print('  outpatient ~0 is CORRECT: no ED stay + no ICU stay => MIMIC-IV records no vitals.')
print('  inpatient < 1 is CORRECT: chartevents is ICU-only; ward patients have none.')

# temperature sanity: after the F->C normalisation the median must be ~37, not ~98
if 'vital_temp_c' in vval:
    med = float(vval['vital_temp_c'].median())
    print(f'\ntemp median after normalisation: {med:.1f} C')
    assert 30 < med < 40, f'temperature median {med:.1f} -- F->C conversion is wrong somewhere'
    print('  OK  Celsius (a ~98 here would mean a Fahrenheit source slipped through)')

assert (vitq['delta_h'] >= 0).all(), 'LEAK: vital measured after index_time'
print('OK  no vital from the future')

## C7-ter · Comorbidities — diabetes, HIV (§A7 extension, prior-discharge anchored)

Not in the plan. Added on Tuan's call. The reason to be careful is **not** that these
features are illegitimate — a chronic diagnosis from a previous discharge is exactly the
kind of thing a clinician *does* know when reading today's film. Two distinct hazards
get conflated, and only the first is fatal:

| | hazard | handled how |
|---|---|---|
| **Leakage** | ICD codes are assigned at **discharge**. Taking the *current* admission's codes uses information that did not exist at `index_time`. | `a.dischtime < c.index_time` — prior, **fully closed** admissions only. Asserted to 0, §E1-style. |
| **Confound** | Patients with thick histories get longer reports; CheXpert labels are NLP-extracted from those reports → more positive labels *independent of the film* (§A4). | **not preventable, and not a reason to exclude** — it is measurable. `como_*` stays independently ablatable so D4-style tests can separate comorbidity from utilization. |

**The mask is doing real work here.** `como_diabetes = 0` is ambiguous on its own: it
means either *"has prior admissions, none coded diabetes"* (evidence of absence) or
*"never admitted before, so nothing could be coded"* (absence of evidence). `como_mask`
= 1 iff the patient has ≥1 prior closed admission — i.e. iff a code *could* have been
observed. Same failure the NO_GROWTH tokens exist to prevent (§A6), same fix.

ICD-9 and ICD-10 both appear in MIMIC-IV and use unrelated code spaces, so each concept
needs both. The codes are **printed before use** — a silently non-matching pattern would
emit an all-zero column and no error.

In [ ]:
COMO = {
  # concept -> (icd9 predicate, icd10 predicate).  Both versions live in MIMIC-IV.
  'diabetes': ("icd_code LIKE '250%'",
               "icd_code LIKE 'E08%' OR icd_code LIKE 'E09%' OR icd_code LIKE 'E10%' "
               "OR icd_code LIKE 'E11%' OR icd_code LIKE 'E13%'"),
  'hiv':      ("icd_code LIKE '042%' OR icd_code = 'V08' OR icd_code LIKE '07953%'",
               "icd_code LIKE 'B20%' OR icd_code = 'Z21'"),
}
_pred = lambda k: (f"((icd_version = 9  AND ({COMO[k][0]})) OR "
                   f" (icd_version = 10 AND ({COMO[k][1]})))")

# ---- VERIFY the patterns hit real codes before trusting them ----
seen = run_q(f"""
SELECT concept, icd_version, icd_code, ANY_VALUE(long_title) AS title, COUNT(*) AS n
FROM (
  {" UNION ALL ".join(f"SELECT '{k}' AS concept, icd_version, icd_code FROM `{HOSP}.diagnoses_icd` WHERE {_pred(k)}" for k in COMO)}
) JOIN `{HOSP}.d_icd_diagnoses` USING (icd_code, icd_version)
GROUP BY concept, icd_version, icd_code
QUALIFY ROW_NUMBER() OVER (PARTITION BY concept, icd_version ORDER BY COUNT(*) DESC) <= 4
ORDER BY concept, icd_version, n DESC
""", 'icd-verify')
print('what the ICD patterns actually matched (top 4 per concept/version):')
print(seen.to_string(index=False))
for k in COMO:
    for v in (9, 10):
        assert ((seen.concept == k) & (seen.icd_version == v)).any(), \
            f'{k} matched NOTHING in ICD-{v} -> the pattern is wrong, not the data'
print('\nOK  every concept matches in both ICD-9 and ICD-10')

# ---- extract, anchored to PRIOR CLOSED admissions only ----
como = run_q(f"""
WITH prior_adm AS (              -- admissions fully finished before the film
  SELECT c.study_id, c.subject_id, a.hadm_id, a.dischtime
  FROM {COH} c JOIN `{HOSP}.admissions` a
    ON a.subject_id = c.subject_id
   AND CAST(a.dischtime AS TIMESTAMP) < c.index_time     -- THE causal anchor
),
dx AS (
  SELECT p.study_id, d.icd_code, d.icd_version
  FROM prior_adm p JOIN `{HOSP}.diagnoses_icd` d USING (hadm_id)
)
SELECT c.study_id,
       CAST(COUNTIF(p.hadm_id IS NOT NULL) > 0 AS INT64)              AS como_mask,
       {", ".join(f"CAST(LOGICAL_OR({_pred(k)}) AS INT64) AS como_{k}" for k in COMO)}
FROM {COH} c
LEFT JOIN prior_adm p USING (study_id)
LEFT JOIN dx        USING (study_id)
GROUP BY c.study_id
""", 'comorbidity')
como = como.fillna(0)
for c in como.columns:
    if c != 'study_id': como[c] = como[c].astype('int8')
print(f'\ncomorbidity rows: {len(como):,}')

# ---- §E1-style leakage gate: no code may come from an open/future admission ----
leak_c = run_q(f"""
SELECT COUNTIF(CAST(a.dischtime AS TIMESTAMP) >= c.index_time) AS n_future_dischtime,
       COUNTIF(a.dischtime IS NULL)                            AS n_open_admission
FROM {COH} c JOIN `{HOSP}.admissions` a
  ON a.subject_id = c.subject_id AND CAST(a.dischtime AS TIMESTAMP) < c.index_time
""", 'como-leak').iloc[0]
assert int(leak_c.n_future_dischtime) == 0 and int(leak_c.n_open_admission) == 0, leak_c.to_dict()
print('LEAKAGE GATE: no comorbidity sourced from an unfinished/future admission  OK')

# ---- coverage: read the mask before reading the rates ----
print(f'\ncomo_mask (has >=1 prior closed admission) : {como.como_mask.mean():.2%}'
      f'   <- the rest CANNOT have any code; that is why the mask ships')
for k in COMO:
    among = como.loc[como.como_mask == 1, f'como_{k}'].mean()
    print(f'  como_{k:9s} {como[f"como_{k}"].mean():6.2%} of cohort  |  {among:6.2%} '
          f'among those with prior admissions')

## C8 · Assemble §A8 schema + train-only standardization

All merges are **LEFT** from the cohort (§A6): a study with no micro must survive
as `MISSING`, not vanish. Standardization stats and vocabs come from **train
only**.

In [ ]:
# C8 consumes what C4 / C7 / C7-bis / C7-ter produced. Without this, a cell that failed
# earlier surfaces here as a bare `NameError: vval`, which sends you debugging the wrong
# cell -- the same stale-kernel trap need() exists for at C0.
for _name, _cell in [('t1','C4 (tiers)'), ('val','C7 (labs)'), ('vval','C7-bis (vitals)'),
                     ('vdlt','C7-bis (vitals)'), ('como','C7-ter (comorbidity)')]:
    if _name not in globals():
        raise RuntimeError(f'`{_name}` is not defined -> run {_cell} first, then this cell.\n'
                           f'    (a cell above almost certainly raised; scroll up to it.)')

df = cohort.copy()
for t in (t1, t2, t3, t4, como):
    df = df.merge(t, on='study_id', how='left')          # LEFT: never drop a study
# comorbidity is already 0/1 per study; a LEFT miss means 'no prior admission' -> 0,
# which como_mask=0 already records. No standardization: these are indicators.
for _c in [c for c in df.columns if c.startswith('como_')]:
    df[_c] = df[_c].fillna(0).astype('int8')

df['t1_org_ids']  = [to_ids(v, org_vocab,  MISSING) for v in df.get('t1_org_itemids')]
df['t1_spec_ids'] = [to_ids(v, spec_vocab, MISSING) for v in df.get('t1_spec_itemids')]
df['t2_spec_ids'] = [to_ids(v, spec_vocab, MISSING) for v in df.get('t2_spec_itemids')]
df['t3_org_ids']  = [to_ids(v, org_vocab,  MISSING) for v in df.get('t3_org_itemids')]

# T1 status -> single categorical id (§A6)
df['t1_status'] = np.select(
    [df.t1_any_positive.fillna(False), df.t1_any_cancelled.fillna(False),
     df.t1_any_no_growth.fillna(False)],
    [1, 2, 3], default=0)                                 # 0 = MISSING (no specimen in-window)

df['t2_pending']        = df['t2_pending'].fillna(False).astype('int8')
df['t2_n_pending']      = df['t2_n_pending'].fillna(0).astype('int16')
df['t2_time_imprecise'] = df['t2_time_imprecise'].fillna(0).astype('int8')
df['t3_n_specimens']    = df['t3_n_specimens'].fillna(0).astype('int16')
df['t1_same_encounter'] = df['t1_same_encounter'].fillna(False).astype('int8')

# §A8 defines t3_recency_bin as {0-7d, 7-30d, 30-90d}. After the remap T1 owns [0,7),
# so the 0-7d bucket is structurally empty -> the bins become {7-30d, 30-90d}.
df['t3_recency_bin'] = pd.cut(df['t3_days_since_last_positive'],
                              [T1_WINDOW_DAYS - 0.001, 30, T3_WINDOW_DAYS],
                              labels=['7-30d','30-90d']).cat.add_categories(
                              'none').fillna('none')
for c in ['t4_n_prior_positive_ever','t4_years_since_first_culture','t4_n_prior_encounters']:
    df[c] = df[c].fillna(0).astype(float)

# ---- value + mask + train-only standardization (§A7 contract) ----
tr = df.split == 'train'
stats = {}

def fit_apply(raw, name, use_log):
    """Winsorize+standardize on TRAIN observed, median-impute, and return (value, mask).
    The mask is returned, never folded away: whether a thing was measured is itself the
    clinical variable (§A7 MNAR)."""
    obs = ~np.isnan(raw)
    x = np.log1p(np.clip(raw, 0, None)) if use_log else raw.copy()
    o = x[tr.to_numpy() & obs]
    if len(o) == 0:
        stats[name] = dict(log=use_log, lo=0, hi=0, med=0, mu=0, sd=1)
    else:
        lo, hi = (-np.inf, np.inf) if use_log else np.percentile(o, WINSOR_PCT)
        oc = np.clip(o, lo, hi)
        stats[name] = dict(log=use_log, lo=float(lo), hi=float(hi), med=float(np.median(oc)),
                           mu=float(oc.mean()), sd=float(oc.std() or 1.0))
    s = stats[name]
    x = np.clip(x, s['lo'], s['hi'])
    x = np.where(obs, x, s['med'])                        # median-impute...
    return ((x - s['mu']) / s['sd']).astype('float32'), obs.astype('float32')   # ...keep mask

for c in LABS:
    raw = val[c].reindex(df.study_id).to_numpy(float) if c in val else np.full(len(df), np.nan)
    df[f'{c}_value'], df[f'{c}_mask'] = fit_apply(raw, c, c in LOG_LABS)
    # `_raw`: the measured value in NATIVE units, NaN where not measured. No winsorize, no
    # log, no z-score, no impute -- so 07 can export true raw without inverting anything
    # (de-standardizing loses the winsorized tails; this does not). `_value` stays for 06.
    df[f'{c}_raw'] = raw.astype('float32')
    d_h = dlt[c].reindex(df.study_id).to_numpy(float) if c in dlt else np.full(len(df), np.nan)
    df[f'{c}_delta_h'] = np.where(np.isnan(d_h), 0.0, d_h).astype('float32')

# vitals: identical §A7 contract (winsorize -> median-impute -> z-score, keep the mask).
# None are logged: they are bounded physiologic quantities, not right-skewed counts, and
# C7-bis already dropped the impossible values rather than clipping them.
for c in VITALS:
    raw = vval[c].reindex(df.study_id).to_numpy(float) if c in vval else np.full(len(df), np.nan)
    df[f'{c}_value'], df[f'{c}_mask'] = fit_apply(raw, c, False)
    df[f'{c}_raw'] = raw.astype('float32')       # native units (bpm, C, mmHg), NaN if not measured
    d_h = vdlt[c].reindex(df.study_id).to_numpy(float) if c in vdlt else np.full(len(df), np.nan)
    df[f'{c}_delta_h'] = np.where(np.isnan(d_h), 0.0, d_h).astype('float32')

# recency: same contract. t1_days_since_last_positive is NULL for ~59% of T1 studies
# (NO_GROWTH/CANCELLED only), which is exactly why it ships with a mask rather than a
# silently-imputed number.
for c in RECENCY:
    raw = df[c].astype(float).to_numpy() if c in df else np.full(len(df), np.nan)
    df[f'{c}_value'], df[f'{c}_mask'] = fit_apply(raw, c, False)
    df = df.drop(columns=[c])

print('lab mask coverage    :', {c: round(float(df[f"{c}_mask"].mean()),3) for c in LABS})
print('recency mask coverage:', {c.replace("_days_since_last",""): round(float(df[f"{c}_mask"].mean()),3)
                                 for c in RECENCY})
print('t1_same_encounter    :', f'{df.t1_same_encounter.mean():.2%} of cohort')

## C9 · §E1 acceptance tests

**Deviation from §E1, deliberate.** The plan guards no-selection-on-EHR with

```python
assert 0.40 <= frac_studies_with_no_micro <= 0.60
```

Two problems, both fatal to that assert:

1. **The metric mixed units.** T1 was counted in *specimens* (`t1_status`), T3 in
   *organisms* (`t3_org_ids`). So a study whose only culture came back **negative**
   was counted as "never cultured" — the §A6 error in miniature, inside the very
   assert meant to protect the data.
2. **The band is a proxy that drifts with our own parameters.** It was calibrated on
   the encounter-based tiers. Widening T1 to `<7d` (any encounter) correctly captures
   fresh no-growth results from other encounters, so the number *fell* — `0.4039 →
   0.3810` — because it became **more accurate**, and the assert fired on that. An
   assert that must be re-tuned every time a window moves protects nothing.

Meanwhile the thing §A1 actually forbids is **structural and checkable**: `01` never
queries microbiology at all, so selection-on-EHR is impossible by construction. We
assert *that* (C), and report the fraction as a diagnostic (A).

In [ ]:
n = len(df)

# ---- (A) no_micro, measured in ONE unit: specimens. A NO_GROWTH culture IS
#          microbiology -- "tested, nothing grew" is a result, not an absence.
no_micro       = ((df.t1_status == 0) & (df.t2_pending == 0)
                  & (df.t3_n_specimens == 0)).mean()
no_micro_mixed = ((df.t1_status == 0) & (df.t2_pending == 0)
                  & df.t3_org_ids.map(lambda v: v == [MISSING])).mean()
print(f'frac_studies_with_no_micro  [specimen unit, {T3_WINDOW_DAYS}d window] = {no_micro:.4f}')
print(f'   the same thing, old mixed-unit formula                  = {no_micro_mixed:.4f}')
print(f'   difference = {no_micro_mixed-no_micro:.4f}  -> studies cultured in-window whose'
      f' every result was negative;\n   the old formula called those "no micro".')
print(f'   (§E1 quotes 0.40-0.60. That band was calibrated on encounter-based tiers and'
      f'\n    drifts with T1_WINDOW_DAYS -> reported, NOT asserted. See (C).)')

# ---- (C) what §A1 actually forbids, asserted structurally ----
# §A1 bans EXCLUDING on EHR. A step that merely *names* EHR is not a violation --
# 01's step 5 is literally '... KEPT (no EHR-based exclusion)', i.e. it names EHR to
# say it does not filter on it. The violation is: references EHR *and dropped rows*.
attr = pd.read_csv(os.path.join(OUT_DIR, 'attrition.csv'))
EHR_WORDS = re.compile(r'micro|culture|organism|specimen|\blab\b|ehr', re.I)
offend = [(s, int(d)) for s, d in zip(attr['step'], attr['n_dropped'])
          if EHR_WORDS.search(str(s)) and d > 0]
assert not offend, f'§A1 VIOLATED -- an EHR-referencing step dropped rows: {offend}'
print(f'\nOK  §A1: no attrition step in 01 dropped rows on an EHR criterion:')
for s, r, dp in zip(attr['step'], attr['n_remaining'], attr['n_dropped']):
    tag = '  <- names EHR, drops 0' if EHR_WORDS.search(str(s)) else ''
    print(f'      {int(dp):>7,} dropped | {int(r):>7,} left | {s}{tag}')

# 02 must not drop studies either: every micro/lab merge is LEFT by construction
assert len(df) == len(cohort), f'02 dropped studies: {len(cohort):,} -> {len(df):,}'
print(f'OK  02 preserved all {len(df):,} studies -> no EHR-based attrition downstream')

k_t1 = df.t1_org_ids.map(lambda v: 0 if v == [MISSING] else len(v))
k_t3 = df.t3_org_ids.map(lambda v: 0 if v == [MISSING] else len(v))
print(f'\norganisms/study  T1: median {k_t1.median():.0f}  mean {k_t1.mean():.2f}  max {k_t1.max()}')
print(f'organisms/study  T3: median {k_t3.median():.0f}  mean {k_t3.mean():.2f}  max {k_t3.max()}')
assert k_t1.mean() < 10 and k_t3.mean() < 10, 'mean near 49 => window not applied (old Cartesian bug)'

# Tier coverage has TWO defensible units and they differ by ~2.4x. Report the
# organism-based one (the definition §E1's 11-16% references came from -- the source
# query carried `WHERE org_name IS NOT NULL`). The specimen-based one is printed too,
# because it is the honest denominator for T2 (which has no organism identity at all)
# and because a NO_GROWTH result is a real finding, not an absence.
cov_org = dict(T1=float((k_t1 > 0).mean()), T3=float((k_t3 > 0).mean()))
cov_spec = dict(T1=float((df.t1_status > 0).mean()), T2=float(df.t2_pending.mean()),
                T3=float((df.t3_n_specimens > 0).mean()),
                T4=float((df.t4_n_prior_positive_ever > 0).mean()))

print('\ntier coverage -- >=1 ORGANISM  [the reported definition, cf. §E1 T1~11%, T3~12-16%]')
for k, v in cov_org.items(): print(f'   {k}: {v:.2%}')
print('\ntier coverage -- >=1 SPECIMEN  [includes NO_GROWTH; T2 is only definable here]')
for k, v in cov_spec.items(): print(f'   {k}: {v:.2%}')
print(f'\n   ratio T1 org/spec = {cov_org["T1"]/max(cov_spec["T1"],1e-9):.2f}'
      f'  -> the rest of T1 is NO_GROWTH/CANCELLED, i.e. "tested, nothing grew"')

print('\nby setting:')
print(df.assign(has_t1=df.t1_status > 0, has_t2=df.t2_pending == 1)
        .groupby('setting')[['has_t1','has_t2']].mean().round(3).to_string())

assert df.groupby('study_id').size().max() == 1
S = {s: set(df.loc[df.split==s,'subject_id']) for s in ['train','val','test']}
for a in S:
    for b in S:
        if a < b: assert not (S[a] & S[b])
print('\nE1 PASSED')

### C9-bis · Marginal tier coverage — the §D3/§D4 ladder

**Total coverage is the wrong number for an ablation.** §A4 quotes Tier 3 as
*"≈12.4% **additional** coverage"* and defines `frac_tier3_ONLY_gain` as studies T3
reaches that T1 does not. Our headline T3 is a **total**; the two are different
quantities and comparing them is meaningless.

This cell reports the cumulative ladder that §D3 actually ablates
(`()` → `(1,2)` → `(1,2,3)` → `(1,2,3,4)`) plus each tier's marginal gain, so
D3/D4 can be read directly off it.

> **Comparability caveat:** the plan's 12.4% was measured on the *encounter-based*
> tiers (T1 = same encounter). Ours are recency-based, so even the marginal figures
> are only approximately comparable — the definitions moved underneath the number.

In [ ]:
u_t1 = k_t1 > 0                                   # organism unit (the reported one)
u_t3 = k_t3 > 0
u_t2 = df.t2_pending == 1                         # T2 has no organisms by construction
u_t4 = df.t4_n_prior_positive_ever > 0

print('cumulative coverage -- the §D3 ablation ladder (organism unit for T1/T3):')
ladder = [('(1)          T1 only',            u_t1),
          ('(1,2)        + pending',          u_t1 | u_t2),
          ('(1,2,3)      + recent history',   u_t1 | u_t2 | u_t3),
          ('(1,2,3,4)    + long-term burden', u_t1 | u_t2 | u_t3 | u_t4)]
prev = None
for name, m in ladder:
    inc = '' if prev is None else f'   (+{m.mean()-prev:.2%})'
    print(f'   {name:<30} {m.mean():7.2%}{inc}'); prev = m.mean()

print('\nmarginal gain of each tier:')
print(f'   T2 beyond T1              : {(u_t2 & ~u_t1).mean():7.2%}')
print(f'   T3 beyond T1+T2           : {(u_t3 & ~(u_t1|u_t2)).mean():7.2%}')
print(f'   T4 beyond T1+T2+T3        : {(u_t4 & ~(u_t1|u_t2|u_t3)).mean():7.2%}')
print(f'   T3 beyond T1  <- §A4 frac_tier3_ONLY_gain, plan quotes 12.4% (90d)')
print(f'                             : {(u_t3 & ~u_t1).mean():7.2%}')
print(f'   T1 and T3 both            : {(u_t1 & u_t3).mean():7.2%}   '
      f'<- overlap; a fresh result AND older history are different facts')

print('\nsame, specimen unit (a NO_GROWTH result is a real finding):')
s_t1, s_t3 = df.t1_status > 0, df.t3_n_specimens > 0
print(f'   T3 beyond T1              : {(s_t3 & ~s_t1).mean():7.2%}')
print(f'   union T1..T4              : {(s_t1|u_t2|s_t3|u_t4).mean():7.2%}'
      f'   (= 1 - {1-(s_t1|u_t2|s_t3|u_t4).mean():.4f} with no micro at all)')

## C10 · Emit §A8 `cohort_study_level.parquet` + vocabs + `class_freq.pkl`

In [ ]:
import pickle
OUT = [c for c in df.columns if not c.endswith('_itemids')
       and c not in ('t1_any_positive','t1_any_no_growth','t1_any_cancelled')]
assert 't1_same_encounter' in OUT and 't1_days_since_last_specimen_value' in OUT
assert 't1_days_since_last_positive_mask' in OUT, 'recency mask missing from the schema'
out = df[OUT].copy()
out['t3_recency_bin'] = out['t3_recency_bin'].astype(str)
out.to_parquet(os.path.join(OUT_DIR,'cohort_study_level.parquet'), index=False)

pickle.dump(dict(org_vocab=org_vocab, spec_vocab=spec_vocab,
                 org_vocab_size=ORG_VOCAB_SIZE, spec_vocab_size=SPEC_VOCAB_SIZE,
                 id2name=id2name, lab_stats=stats, labs=LABS, labels=LABEL_COLS,
                 reserved=dict(PAD=PAD, UNK=UNK, MISSING=MISSING, NO_GROWTH=NO_GROWTH,
                               CANCELLED=CANCELLED, NEGATIVE=NEGATIVE),
                 t1_window_days=T1_WINDOW_DAYS, t3_window_days=T3_WINDOW_DAYS,
                 lab_window_h=LAB_WINDOW_H, recency=RECENCY,
                 vitals=VITALS, comorbidity=[f'como_{k}' for k in COMO],
                 vital_ranges=VIT_RANGE,
                 tier_split='recency'),          # NOT encounter -- see C4
            open(os.path.join(OUT_DIR,'ehr_vocab.pkl'),'wb'))

# class_freq is the ONLY input DB Loss (§B8) takes from this notebook, and a wrong
# one trains quietly. Prove it is a label matrix before writing it.
y = df.loc[df.split=='train', LABEL_COLS].to_numpy()
assert y.shape[1] == 13, f'class_freq would have {y.shape[1]} classes, not 13'
assert set(np.unique(y)).issubset({0, 1}), \
    f'labels are not binary -> a non-label column leaked into LABEL_COLS: ' \
    f'{sorted(set(np.unique(y)))[:8]}'
cfreq = y.sum(0).astype(int)
assert (cfreq > 0).all(), \
    f'label(s) with zero train positives -> DB Loss divides by zero: ' \
    f'{[LABEL_COLS[i] for i in np.where(cfreq == 0)[0]]}'
pickle.dump(dict(class_freq=cfreq, neg_class_freq=(len(y)-y.sum(0)).astype(int),
                 num_train=len(y), labels=LABEL_COLS),
            open(os.path.join(OUT_DIR,'class_freq.pkl'),'wb'))
print(f'class_freq: {dict(zip(LABEL_COLS, cfreq.tolist()))}')
print(f'num_train : {len(y):,}')

print('emitted:', out.shape)
for f in sorted(os.listdir(OUT_DIR)):
    print(' ', f, round(os.path.getsize(os.path.join(OUT_DIR,f))/1e6,2),'MB')
print('\nNEXT: 03_cxr_embeddings.ipynb')